In [2]:
import numpy as np
import serial
import time

In [3]:
def parse_data(l:str):
    if len(l) < 10: return "", []
    if l[:5] != "Data:": return "", []
    s = l.split('\t')

    if len(s) < 2: return "", []
    _t = s[0].split(',')
    if len(_t) < 2: return "", []
    _type = _t[0][5:]
    _size = int(_t[1][7:])
    arr = np.fromstring(s[1], sep = ',', dtype=np.uint32)
    if _type == "ENV":
        env_type = np.bitwise_and(arr, 0xF000) >> 12
        env_read = np.bitwise_and(arr, 0x0FFF)
        env_time = np.bitwise_and(arr, 0xFFFF0000) >> 10
        leaftemp_t = env_time[env_type == 4]
        leaftemp = env_read[env_type == 4] / 10
        timestemp_t = env_time[env_type < 4]
        timestemp = env_type[env_type < 4]
        return "ENV", [timestemp_t, timestemp, leaftemp_t, leaftemp]
    elif _type in ["Fluo", "Fluoref", "SUN", "leaf", "730", "730ref"]:
        return _type, [arr]
    else:
        print(_type)

    return "", []

In [4]:
a = 1
b = 200
c = 20
arr = np.array([a, 0, 1, 0, 0, b, 0, 1, 
                a, 0, 1, 0, 0, b, c, 1,
                a, 0, 1, 0, 0, b, 0, 1], dtype=np.uint8)
cmd = 'arrun,3,' + np.array2string(arr, separator=',', max_line_width = 1000)[1:-1].replace(' ','') + ',\n'
cmd

'arrun,3,1,0,1,0,0,200,0,1,1,0,1,0,0,200,20,1,1,0,1,0,0,200,0,1,\n'

In [11]:
arr.reshape((-1, 8))

array([[  1,   0,   1,   0,   0, 200,   0,   1],
       [  1,   0,   1,   0,   0, 200,  20,   1],
       [  1,   0,   1,   0,   0, 200,   0,   1]], dtype=uint8)

In [5]:
timeout = 100
s = ''
_l = 0
result = {}



with serial.Serial("COM6", baudrate=115200) as ser:
    t0 = time.time()
    ser.write(cmd.encode())
    while time.time() - t0 < timeout:
        if ser.in_waiting > 1:
            c = ser.read()
            if (c > bytes([0])) and (c < bytes([128])):
                s += c.decode()
                _l += 1
                if (c == b"\n") or (c == b"\r"):
                    l = s
                    s = ''
                    _l = 0
                    if l[:-1] == "BAD COMMAND":
                        timeout = 1
                        continue
                    if l[:-1] == "Data sent":
                        timeout = .1
                        continue
                    d_type, data = parse_data(l)
                    if d_type == "ENV":
                        result.update({"env_stamp_t": data[0], "env_stamp": data[1], "env_leaf_t": data[2], "env_leaf": data[3]})
                    elif d_type != "":
                        result.update({d_type: data[0]})



In [8]:
result['Fluo'] / result['Fluoref']

C:\Users\hjcbj\AppData\Local\Temp\ipykernel_16260\1349986147.py:1: RuntimeWarning: invalid value encountered in divide
  result['Fluo'] / result['Fluoref']


array([0.00865759, 0.00920368, 0.01009676, 0.01000342, 0.01017616,
       0.00991962, 0.01008892, 0.01025904, 0.00983074, 0.01008892,
       0.01000684, 0.01017616, 0.01000428, 0.01000513, 0.01009151,
       0.01026167, 0.01017442, 0.01009064, 0.01017703, 0.01009151,
       0.00991792, 0.0100077 , 0.01034365, 0.01025992, 0.00991962,
       0.0101779 , 0.0102608 , 0.01000428, 0.01000599, 0.01034984,
       0.00991877, 0.01000342, 0.01017529, 0.01000599, 0.01017442,
       0.01000513, 0.00983747, 0.01009064, 0.01017703, 0.00992047,
       0.01000513, 0.00992047, 0.01000428, 0.01009064, 0.01008892,
       0.01008978, 0.01034719, 0.00991877, 0.00991877, 0.01009237,
       0.00992047, 0.01009151, 0.01008892, 0.01017703, 0.01034542,
       0.01009064, 0.01008978, 0.01009064, 0.01017355, 0.00983494,
       0.00991962, 0.01000428, 0.0102608 , 0.01017703, 0.01017529,
       0.01009151, 0.01026255, 0.01009237, 0.01009151, 0.01017442,
       0.01009064, 0.00991962, 0.01026343, 0.01000342, 0.01025